# Global Cybersecurity Threats (2015–2024) — Data Cleaning

Load the dataset, inspect it, and apply cleaning steps.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

#CSV_PATH = Path(r"c:\Users\wbhat\Downloads\archive\Global_Cybersecurity_Threats_2015-2024.csv")
CSV_PATH = "./Global_Cybersecurity_Threats_2015-2024.csv"
#out_path = Path(r"c:\Users\wbhat\Downloads\archive\Global_Cybersecurity_Threats_2015-2024_cleaned.csv")
out_path = "./Global_Cybersecurity_Threats_2015-2024_cleaned.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)

## 1. Inspect structure and missing values

In [ ]:
df.info()

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing[missing > 0]

## 2. Check for duplicates

In [ ]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")
if dupes > 0:
    df[df.duplicated(keep=False)].sort_values(df.columns.tolist())

## 3. Clean: drop duplicates and trim whitespace

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()

# Strip whitespace from string columns
for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = df_clean[col].str.strip()

print(f"Rows after dropping duplicates: {len(df_clean):,}")
df_clean.head()

## 4. Validate categorical columns (unique values)

In [ ]:
cat_cols = ["Country", "Attack Type", "Target Industry", "Attack Source", 
            "Security Vulnerability Type", "Defense Mechanism Used"]
for col in cat_cols:
    print(f"{col}: {df_clean[col].nunique()} unique — {sorted(df_clean[col].unique().tolist())}")

## 5. Numeric sanity checks (outliers / invalid ranges)

In [ ]:
df_clean.describe()

In [ ]:
# Year should be 2015–2024
print("Year range:", df_clean["Year"].min(), "-", df_clean["Year"].max())
# Financial loss and resolution time should be non-negative
print("Financial Loss < 0:", (df_clean["Financial Loss (in Million $)"] < 0).sum())
print("Number of Affected Users < 0:", (df_clean["Number of Affected Users"] < 0).sum())
print("Incident Resolution Time (in Hours) < 0:", (df_clean["Incident Resolution Time (in Hours)"] < 0).sum())

## 6. Save cleaned dataset

In [ ]:
df_clean.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
df_clean.shape

# Data Visualizations
Exploring the cleaned dataset using Matplotlib.

In [ ]:
import os
import matplotlib.pyplot as plt
df_plot = df_clean.copy()

os.makedirs('figures', exist_ok=True)


## Number of Incidents by Attack Type

In [ ]:
attack_counts = df_plot['Attack Type'].value_counts()
plt.figure(figsize=(10, 6))
attack_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Number of Incidents by Attack Type', fontsize=14)
plt.xlabel('Attack Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figures/attack_type_incidents.png', bbox_inches='tight')
plt.show()

## Total Financial Loss by Target Industry

In [ ]:
loss_by_industry = df_plot.groupby('Target Industry')['Financial Loss (in Million $)'].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
loss_by_industry.plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Total Financial Loss by Target Industry (2015-2024)', fontsize=14)
plt.xlabel('Target Industry', fontsize=12)
plt.ylabel('Total Financial Loss (Millions $)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figures/financial_loss_industry.png', bbox_inches='tight')
plt.show()

## Combined: Incidents and Financial Loss Over Time

In [ ]:
year_counts = df_plot['Year'].value_counts().sort_index()
loss_by_year = df_plot.groupby('Year')['Financial Loss (in Million $)'].sum().sort_index()

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Number of Incidents (left y-axis)
color1 = 'tab:green'
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Number of Incidents', color=color1, fontsize=12)
ax1.plot(year_counts.index, year_counts.values, marker='o', linestyle='-', color=color1, linewidth=2, label='Incidents')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_xticks(year_counts.index)

# Create a twin axis for Financial Loss (right y-axis)
ax2 = ax1.twinx()
color2 = 'tab:purple'
ax2.set_ylabel('Total Financial Loss (Millions $)', color=color2, fontsize=12)
ax2.plot(loss_by_year.index, loss_by_year.values, marker='s', linestyle='--', color=color2, linewidth=2, label='Financial Loss')
ax2.tick_params(axis='y', labelcolor=color2)

# Add titles and layout
plt.title('Cybersecurity Incidents and Financial Loss Over Time', fontsize=14)
fig.tight_layout()
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.savefig('figures/incidents_loss_time.png', bbox_inches='tight')
plt.show()

## Top 10 Countries by Number of Affected Users

In [ ]:
users_by_country = df_plot.groupby('Country')['Number of Affected Users'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
users_by_country.sort_values().plot(kind='barh', color='orange', edgecolor='black')
plt.title('Top 10 Countries by Number of Affected Users', fontsize=14)
plt.xlabel('Total Affected Users', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.tight_layout()
plt.savefig('figures/top10_affected_users.png', bbox_inches='tight')
plt.show()

## Distribution of Attack Sources

In [ ]:
source_counts = df_plot['Attack Source'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(source_counts, labels=source_counts.index, autopct='%1.1f%%', startangle=140, colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
plt.title('Distribution of Attack Sources', fontsize=14)
plt.savefig('figures/attack_sources_distribution.png', bbox_inches='tight')
plt.show()

## Incidents by Security Vulnerability Type

In [ ]:
vuln_counts = df_plot['Security Vulnerability Type'].value_counts()

plt.figure(figsize=(10, 6))
vuln_counts.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Incidents by Security Vulnerability Type', fontsize=14)
plt.xlabel('Vulnerability Type', fontsize=12)
plt.ylabel('Number of Incidents', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/security_vulnerability_types.png', bbox_inches='tight')
plt.show()

In [ ]:
# Financial Loss by Attack Source
loss_by_source = df_plot.groupby('Attack Source')['Financial Loss (in Million $)'].sum().sort_values()

plt.figure(figsize=(10, 6))
loss_by_source.plot(kind='barh', color='coral', edgecolor='black')
plt.title('Total Financial Loss by Attack Source', fontsize=14)
plt.xlabel('Total Financial Loss (Millions $)', fontsize=12)
plt.ylabel('Attack Source', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/financial_loss_attack_source.png', bbox_inches='tight')
plt.show()

In [ ]:
# Financial Loss by Security Vulnerability Type
loss_by_vuln = df_plot.groupby('Security Vulnerability Type')['Financial Loss (in Million $)'].sum().sort_values()

plt.figure(figsize=(10, 6))
loss_by_vuln.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Total Financial Loss by Security Vulnerability Type', fontsize=14)
plt.xlabel('Total Financial Loss (Millions $)', fontsize=12)
plt.ylabel('Security Vulnerability Type', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/financial_loss_vulnerability_type.png', bbox_inches='tight')
plt.show()

In [ ]:
# Financial Loss by Country
loss_by_country = df_plot.groupby('Country')['Financial Loss (in Million $)'].sum().sort_values()

plt.figure(figsize=(10, 6))
loss_by_country.plot(kind='barh', color='gold', edgecolor='black')
plt.title('Total Financial Loss by Country', fontsize=14)
plt.xlabel('Total Financial Loss (Millions $)', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/financial_loss_country.png', bbox_inches='tight')
plt.show()

In [ ]:
# Incident Resolution Time by Attack Type (Average)
res_by_attack = df_plot.groupby('Attack Type')['Incident Resolution Time (in Hours)'].mean().sort_values()

plt.figure(figsize=(10, 6))
res_by_attack.plot(kind='barh', color='lightcoral', edgecolor='black')
plt.title('Average Incident Resolution Time by Attack Type', fontsize=14)
plt.xlabel('Average Resolution Time (Hours)', fontsize=12)
plt.ylabel('Attack Type', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/resolution_time_attack_type.png', bbox_inches='tight')
plt.show()

In [ ]:
# Incident Resolution Time by Target Industry (Average)
res_by_industry = df_plot.groupby('Target Industry')['Incident Resolution Time (in Hours)'].mean().sort_values()

plt.figure(figsize=(10, 6))
res_by_industry.plot(kind='barh', color='lightseagreen', edgecolor='black')
plt.title('Average Incident Resolution Time by Target Industry', fontsize=14)
plt.xlabel('Average Resolution Time (Hours)', fontsize=12)
plt.ylabel('Target Industry', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/resolution_time_target_industry.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of Financial Loss by Year (Boxplots)
plt.figure(figsize=(12, 7))
df_plot.boxplot(column='Financial Loss (in Million $)', by='Year', ax=plt.gca(), grid=False, 
                patch_artist=True, boxprops=dict(facecolor='lightblue', color='black'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='o', markerfacecolor='black', alpha=0.5))
plt.title('Distribution of Financial Loss per Year', fontsize=14)
plt.suptitle('')  # Suppress the default title from pandas boxplot to make it cleaner
plt.xlabel('Year', fontsize=12)
plt.ylabel('Financial Loss (Millions $)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('figures/financial_loss_per_year_boxplot.png', bbox_inches='tight')
plt.show()

# Data Transformation & Normalization
Preparing the continuous variables for pattern mining. We will normalize the continuous variables and discretize them so that they can be used with Apriori/FP-Growth algorithms which require categorical/binary inputs.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Create a copy for transformation
df_trans = df_clean.copy()

## Normalization (Min-Max Scaling)
Scaling if we need these features for distance-based ML algorithms later.

In [ ]:
scaler = MinMaxScaler()
num_cols = ['Financial Loss (in Million $)', 'Number of Affected Users', 'Incident Resolution Time (in Hours)']

df_trans[[f"{c}_Scaled" for c in num_cols]] = scaler.fit_transform(df_trans[num_cols])

print("Min-Max Scaled Data Preview:")
df_trans[[f"{c}_Scaled" for c in num_cols]].head()

## Discretization (Binning)
Apriori and FP-Growth require categorical or boolean data. We convert continuous ranges into discrete categories.

In [ ]:
# Binning Financial Loss (Low, Medium, High)
df_trans['Financial_Loss_Category'] = pd.qcut(df_trans['Financial Loss (in Million $)'], q=3, labels=['Low', 'Medium', 'High'])

# Binning Affected Users (Low, Medium, High)
df_trans['Affected_Users_Category'] = pd.qcut(df_trans['Number of Affected Users'], q=3, labels=['Low', 'Medium', 'High'])

# Binning Resolution Time (Fast, Average, Slow)
df_trans['Resolution_Time_Category'] = pd.qcut(df_trans['Incident Resolution Time (in Hours)'], q=3, labels=['Fast', 'Average', 'Slow'])

print("Discretized Data Preview:")
df_trans[['Financial_Loss_Category', 'Affected_Users_Category', 'Resolution_Time_Category']].head()

## Drop original continuous columns

In [ ]:
df_pattern_mining = df_trans.drop(columns=num_cols + [f"{c}_Scaled" for c in num_cols])

print("Dataset ready for Binary Variable Creation (One-Hot Encoding):")
df_pattern_mining.head()

## Binary Variable Creation (One-Hot Encoding for Pattern Mining)

In [ ]:
"""
Purpose:
    Convert categorical and discretized variables into binary (one-hot) columns
    so that the dataset is suitable for Apriori and FP-Growth pattern mining.

Method:
    One-hot encoding via pandas get_dummies() with drop_first=False and
    prefix_sep="="; result cast to boolean.

Parameters:
    drop_first=False, prefix_sep="=".

Structural impact:
    Produces a binary matrix (e.g., 3000 rows × 46 columns) with one column per
    category level; each cell is True/False.

Key results:
    Binary transaction matrix ready for frequent itemset mining.
"""
df_binary = pd.get_dummies(
    df_pattern_mining.select_dtypes(include=["object", "category"]),
    drop_first=False,
    prefix_sep="="
)
df_binary = df_binary.astype(bool)
print("Shape:", df_binary.shape)
df_binary.head()

## Validation Checks

In [ ]:
"""
Purpose:
    Verify the binary transaction matrix has no invalid values, missing data, or
    empty rows before running frequent itemset algorithms.

Method:
    Check unique values (expect only True/False), total missing values (expect 0),
    and count of rows with row sum == 0 (expect 0).

Parameters:
    None (inspection only).

Structural impact:
    No change to shape; validates existing structure (e.g., 3000 rows × 46 columns).

Key results:
    Confirmation of valid binary matrix (no NaN, no all-zero rows).
"""
print("Unique values:", pd.unique(df_binary.values.ravel()))
print("Total missing values:", df_binary.isna().sum().sum())
print("Number of rows where row sum == 0:", (df_binary.sum(axis=1) == 0).sum())

## Frequent Itemsets with Apriori and FP-Growth (min_support = 0.05)

In [ ]:
"""
Purpose:
    Discover frequent itemsets in the binary transaction matrix using both Apriori
    and FP-Growth for comparison and downstream association rule mining.

Method:
    Apriori and FP-Growth from mlxtend.frequent_patterns with use_colnames=True.

Parameters:
    min_support = 0.05.

Structural impact:
    Input: binary matrix (e.g., 3000 rows × 46 columns). Output: itemset tables
    (e.g., 292 frequent itemsets per algorithm).

Key results:
    292 frequent itemsets identified at min_support = 0.05; Apriori and FP-Growth
    yield identical itemsets; FP-Growth used for association rules.
"""
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

min_support = 0.05
freq_apriori = apriori(df_binary, min_support=min_support, use_colnames=True)
freq_fpgrowth = fpgrowth(df_binary, min_support=min_support, use_colnames=True)

print("Apriori itemsets:", len(freq_apriori))
print("FP-Growth itemsets:", len(freq_fpgrowth))
print("\nTop 10 Apriori (by support):")
display(freq_apriori.sort_values("support", ascending=False).head(10))
print("\nTop 10 FP-Growth (by support):")
display(freq_fpgrowth.sort_values("support", ascending=False).head(10))

## Association Rules (from FP-Growth itemsets)

In [ ]:
"""
Purpose:
    Generate association rules from frequent itemsets to quantify implications
    (antecedent → consequent) via support, confidence, and lift.

Method:
    association_rules() from mlxtend on FP-Growth itemsets; metric="lift",
    min_threshold=1.0; antecedents/consequents formatted as strings for display.

Parameters:
    metric="lift", min_threshold=1.0.

Structural impact:
    One rule table with columns including antecedents, consequents, support,
    confidence, lift. No change to original dataframe shape.

Key results:
    286 rules generated; rules sorted by lift and confidence for interpretation.
"""
from IPython.display import display

rules = association_rules(freq_fpgrowth, metric="lift", min_threshold=1.0)
rules_sorted = rules.sort_values(by=["lift", "confidence"], ascending=[False, False])
rules_sorted["antecedents"] = rules_sorted["antecedents"].apply(lambda x: ", ".join(list(x)))
rules_sorted["consequents"] = rules_sorted["consequents"].apply(lambda x: ", ".join(list(x)))
display(rules_sorted[["antecedents", "consequents", "support", "confidence", "lift"]].head(15))
print("Total rules generated:", len(rules))